# k-Nearest Neighbour para decidir cuál modelo es mejor para cada consulta

In [100]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
import joblib
import gower
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score

In [101]:
data = "mejores_resultados.json"
data = pd.read_json(data)

In [102]:
print(data.head())

  departamento                                              query  \
0    marketing  Redacta una secuencia de 5 correos electrónico...   
1    marketing  Escribe el guion completo para un vídeo de You...   
2    marketing  Redacta una nota de prensa formal de 500 palab...   
3    marketing  Genera 10 ideas de ganchos (hooks) visuales y ...   
4    marketing  Crea 3 variaciones de copy publicitario para a...   

                  model  latency_s  prompt_tokens  concretitud  \
0  llama-3.1-8b-instant   1.215854             80          0.0   
1  llama-3.1-8b-instant   0.941735             79          0.0   
2  llama-3.1-8b-instant   1.217726             73          0.0   
3           llama3.2:3b   3.083361             65          0.0   
4           llama3.2:3b   2.365051             75          0.0   

   especificacion  criticidad  tamano_respuesta  
0            4.32         0.0              8.03  
1            4.30         0.0             10.00  
2            4.06         0.0         

In [103]:
data.columns

Index(['departamento', 'query', 'model', 'latency_s', 'prompt_tokens',
       'concretitud', 'especificacion', 'criticidad', 'tamano_respuesta'],
      dtype='str')

In [104]:
# Crear el DataFrame con las variables predictoras
datos = data.drop(columns=['latency_s', 'query'])
datos['target'] = data['model']

display(datos.head())  

print(f"Número de ejemplos (filas): {datos.shape[0]}")
print(f"Número de variables totales (columnas): {datos.shape[1]}")

columnas_numericas = datos.select_dtypes(include=[np.number]).columns
medianas = datos[columnas_numericas].median()
datos[columnas_numericas] = datos[columnas_numericas].fillna(medianas)
columnas_string_conflictivas = datos.select_dtypes(include=['string']).columns

for col in columnas_string_conflictivas:
    datos[col] = datos[col].astype(object)

,departamento,model,prompt_tokens,concretitud,especificacion,criticidad,tamano_respuesta,target
0,marketing,llama-3.1-8b-instant,80,0.0,4.32,0.0,8.03,llama-3.1-8b-instant
1,marketing,llama-3.1-8b-instant,79,0.0,4.30,0.0,10.00,llama-3.1-8b-instant
2,marketing,llama-3.1-8b-instant,73,0.0,4.06,0.0,10.00,llama-3.1-8b-instant
3,marketing,llama3.2:3b,65,0.0,1.50,0.0,5.00,llama3.2:3b
4,marketing,llama3.2:3b,75,0.0,4.12,0.0,2.70,llama3.2:3b


Número de ejemplos (filas): 59
Número de variables totales (columnas): 8


In [105]:
# Separamos características (X) y objetivo (y)
y = datos['target']
X = datos.drop(columns=['target'])

# Dividimos en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y
)

# Comprobamos la distribución (en porcentaje)
print("Distribución en Entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución en Prueba:")
print(y_test.value_counts(normalize=True) * 100)

Distribución en Entrenamiento:
target
llama-3.1-8b-instant    46.808511
llama3.2:3b             46.808511
mistral:7b               6.382979
Name: proportion, dtype: float64

Distribución en Prueba:
target
llama3.2:3b             50.000000
llama-3.1-8b-instant    41.666667
mistral:7b               8.333333
Name: proportion, dtype: float64


In [106]:
X_train_dist = gower.gower_matrix(X_train)

# Inicializamos el modelo k-NN con k=3 y distancia de Gower
knn = KNeighborsClassifier(n_neighbors=3, metric='precomputed')

# 1. Identificamos cuáles son las columnas de texto/categóricas
columnas_categoricas = datos.select_dtypes(include=['object', 'string']).columns

# (Asegúrate de calcular tu 'cat_mask' y tus 'rangos' basándote en el 
# DataFrame original ANTES de aplicar la transformación del siguiente paso).

# 2. Creamos una copia de los datos para no machacar los originales
datos_numericos = datos.copy()

# Aplicamos ordinal encoding (ej: 'marketing' -> 0, 'desarrollo' -> 1)
codificador = OrdinalEncoder()
datos_numericos[columnas_categoricas] = codificador.fit_transform(datos[columnas_categoricas])

matriz_distancias_train = gower.gower_matrix(X_train) 
knn = KNeighborsClassifier(n_neighbors=3, metric='precomputed')
knn.fit(matriz_distancias_train, y_train)

print("¡Modelo entrenado superando el bloqueo de texto!")

¡Modelo entrenado superando el bloqueo de texto!


In [107]:
X_test_dist = gower.gower_matrix(X_test, X_train)

# Generamos las predicciones sobre el conjunto de prueba
y_pred = knn.predict(X_test_dist)

print("Matriz de entrenamiento mapeada con éxito.")
print(f"Predicciones del conjunto de prueba: {y_pred}")

# Contamos cuántas predicciones coinciden exactamente con la realidad
aciertos = (y_test == y_pred).sum()
total_casos = len(y_test)

print(f"El modelo ha acertado {aciertos} de un total de {total_casos} casos.")
porcentaje_aciertos = accuracy_score(y_test, y_pred)
print(f"Porcentaje de aciertos: {porcentaje_aciertos * 100:.2f}%")

Matriz de entrenamiento mapeada con éxito.
Predicciones del conjunto de prueba: ['llama3.2:3b' 'llama3.2:3b' 'llama3.2:3b' 'llama-3.1-8b-instant'
 'llama-3.1-8b-instant' 'mistral:7b' 'llama3.2:3b' 'llama-3.1-8b-instant'
 'llama-3.1-8b-instant' 'llama3.2:3b' 'llama3.2:3b' 'llama-3.1-8b-instant']
El modelo ha acertado 12 de un total de 12 casos.
Porcentaje de aciertos: 100.00%


In [108]:
joblib.dump(knn, 'modelo_knn_ej8.joblib')
print("¡Modelo exportado correctamente!")

¡Modelo exportado correctamente!
